## До того как вы приступите к решению:
**Tools → Settings → Editor → completions / suggestions / linting → disable**

## Задание 1

Допишите 2 реализации функции `increment()`, которая **увеличивает глобальную переменную `counter` на 1**:

**с/без** (!) python синтаксического сахара. Сигнатуру функции менять нельзя.

**1.1: с python синтаксическим сахаром**

In [1]:
counter = 0

def increment():
    global counter
    counter += 1

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


**1.2: без python синтаксического сахара**

In [2]:
counter = 0

def increment():
    globals().__setitem__('counter', int.__add__(globals()['counter'], 1))

increment()
increment()
assert counter == 2, 'try again'
print(f'{counter=} -- great!')

counter=2 -- great!


## Задание 2

Достаньте **только функцию `sqrt`** из модуля `math` и исполните sqrt(169).  
Нельзя исполнять `import math`.

Подготовьте 2 решения.

...

In [3]:
from math import sqrt

result = sqrt(169)
assert result == 13

In [4]:
sqrt = __import__('math',fromlist=["sqrt"]).sqrt

result = sqrt(169)
assert result == 13

## Задание 3

Динамический импорт и перезагрузка.

1. Создайте модуль `mod.py`:

In [5]:
%%writefile mod.py
msg = "A"

Overwriting mod.py


2. Импортируйте его и выведите `msg`

3. Измените `msg` на `B` в файле

4. Без перезагрузки сессии ноутбука, выведите новое значение `msg`

In [6]:
import mod
print(mod.msg)

A


In [7]:
%%writefile mod.py
msg="B"

Overwriting mod.py


In [8]:
import importlib
importlib.invalidate_caches()
mod = importlib.reload(mod)

print(mod.msg)

B


## Задание 4

У вас есть дирректория `pkg`:

In [1]:
!mkdir -p pkg

In [2]:
%%writefile pkg/m1.py
pi = 3.1415_92_65
_e = 2.7
__i = -1

Overwriting pkg/m1.py


```
pkg/
└── m1.py
```

Ниже ячейки для вашего кода, а после задание

### **Первый способ**: через модуль из стандартной библиотеки CPython

In [11]:
from pathlib import Path

Path("pkg/__init__.py").write_text("from .m1 import *\n", encoding="utf-8")

18

### **Второй способ**: в одну строчку без доп.модулей

In [12]:
open("pkg/__init__.py", "w",encoding="utf-8").write("from .m1 import *\n")

18

### Текст задания:
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.  
Вам необходимо изменить структуру `pkg` пакета / содержимое его модулей, чтобы следующий код выполнялся корректно:

In [13]:
from pkg import *
pi

3.14159265

**Важно!**  
При обновлении любых данных в дирректории проекта, вам необходимо перезагружать сессию ipynb:  
`Runtime --> Restart Session` и перезапустить необходимые ячейки задания,  
иначе результаты могут быть для вас некорректными.

## Задание 5

При правильно решённом **задании 4** вам необходимо:
- изменить `pkg`
- дописать код ниже

так, чтобы "дотянуться" до `__i`.  
Нельзя пересоздавать значения `pi`, `_e`, `__i`  и использовать их переменные напрямую в импорте.    
Если вы решите перезагрузить сессию, то для решения **задания 5** необходимо перезапустить ячейки **задания 4**.  

In [3]:
open("pkg/__init__.py", "w", encoding="utf-8").write(
    "from .m1 import *\n"
    "from . import m1 as _m1\n"
    "__i = getattr(_m1, '__i')\n"
    "__all__ = ['pi', '__i']\n"
)

92

### Решение

In [4]:
from pkg import *
__i

-1

## Задание 6

Изменяемое замыкание. Почему этот код ведёт себя неожиданно? Исправьте.

In [8]:
def create_accumulators():
    accs = []
    def make_accumulator():
      total = 0
      def accumulator(x):
        nonlocal total
        total += x
        return total
      return accumulator
    for i in range(3):
      accs.append(make_accumulator())
    return accs

acc_list = create_accumulators()
print(acc_list[0](10))  # Ожидается 10, но получается ошибка

10


## Задание 7
При перезапуске сессии ноутбука решение задачи начинается сначала

### 7.1: Востановите работу `print`, не используя `del`

In [9]:
print = 1

In [10]:
import builtins

print = builtins.print
print("hello")

hello


### 7.1: Удалите объект `print`, после востановите его функционал

In [11]:
print = 1

del print
print("hello")

hello


## Задание 8

Замыкание с изменяемым состоянием. Создайте функцию-счётчик, которая запоминает количество вызовов между разными экземплярами:

In [12]:
def make_shared_counter(shared={'count': 0}):
    def counter():
      shared["count"]+=1
      return shared["count"]
    return counter

c1 = make_shared_counter()
c2 = make_shared_counter()

print(c1())
print(c2())
print(c1())

1
2
3


## Задание 9

Допишите код, чтобы функция `outer` возвращала **словарь с тремя замыканиями**: `add()`, `mul()`, `get()` — работающими с одной и той же закрытой переменной `value`.

In [13]:
def outer(val=0):
    value = val
    def add(x):
      nonlocal value
      value += x
    def mul(x):
      nonlocal value
      value *= x
    def get():
      return value
    return {"add":add,"mul":mul,"get":get}


obj = outer(10)
obj['add'](5)
obj['mul'](2)
assert obj['get']() == 30

## Задание 10

Создать closure, которая принимает функцию и возвращает новую функцию с кэшированием результатов (мемоизацией).

*Теоретическая справка:*

Функция `memoize` принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

В примере с функцией `fib` (числа Фибоначчи) мемоизация уменьшает количество рекурсивных вызовов с экспоненциального до линейного, так как результаты для каждого n вычисляются один раз.

Таким образом, мемоизация экономит время за счёт памяти — класическая оптимизация "время против памяти" — и особенно полезна для функций с дорогими вычислениями и повторяющимися входами.

Алгоритм для решения:

- Функция memoize принимает другую функцию func и создаёт внутри closure, где есть словарь cache для хранения результатов.

- Внутренняя функция wrapper проверяет, есть ли для данного входного аргумента (x) уже вычисленный результат в словаре cache.

- Если результат есть, то он возвращается из кеша, и вычисления не повторяются.

- Если нет, то вызывается исходная функция func(x), результат сохраняется в cache и возвращается.


In [15]:
def memoize(func):
    cache = {}

    def wrapper(x):
        if x in cache:
            return cache[x]

        result = func(x)
        cache[x] = result
        return result

    return wrapper

@memoize
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(10))  # 55

55
